In [12]:
def check_winner(board):
    # board est une liste de 9 cases: 1 pour X, -1 pour O, 0 pour vide
    win_conditions = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
    for qc in win_conditions:
        if board[qc[0]] == board[qc[1]] == board[qc[2]] != 0:
            return board[qc[0]] # Retourne 1 (X gagne) ou -1 (O gagne)
    if 0 not in board:
        return 0 # Match nul
    return None # Partie en cours

def minimax(board, is_maximizing, alpha=-float('inf'), beta=float('inf')):
    res = check_winner(board)
    if res is not None:
        return res
    
    if is_maximizing:
        best_score = -float('inf')
        for i in range(9):
            if board[i] == 0:
                board[i] = 1 # X joue
                score = minimax(board, False, alpha, beta)
                board[i] = 0
                best_score = max(score, best_score)
                alpha = max(alpha, best_score)
                if beta <= alpha: break
        return best_score
    else:
        best_score = float('inf')
        for i in range(9):
            if board[i] == 0:
                board[i] = -1 # O joue
                score = minimax(board, True, alpha, beta)
                board[i] = 0
                best_score = min(score, best_score)
                beta = min(beta, best_score)
                if beta <= alpha: break
        return best_score

In [13]:
def encode_board(board):
    # Transforme [1, -1, 0...] en [1, 0, 0, 1, 0, 0...] (18 features)
    encoded = []
    for cell in board:
        if cell == 1: # X occupe
            encoded.extend([1, 0])
        elif cell == -1: # O occupe
            encoded.extend([0, 1])
        else: # Vide
            encoded.extend([0, 0])
    return encoded

def generate_all_states(board, turn):
    states = set()
    
    def backtrack(current_board, current_turn):
        state_tuple = tuple(current_board)
        # On ne garde que les états où c'est au tour de X [cite: 22, 48]
        if current_turn == 1:
            states.add(state_tuple)
            
        res = check_winner(current_board)
        if res is not None: return
        
        for i in range(9):
            if current_board[i] == 0:
                current_board[i] = current_turn
                backtrack(current_board, -current_turn)
                current_board[i] = 0
                
    backtrack(board, turn)
    return states

In [ ]:
import pandas as pd
import os
if not os.path.exists('ressources'):
    os.makedirs('ressources')

def create_csv():
    all_states = generate_all_states([0]*9, 1) # Commence par X
    data = []
    
    for state in all_states:
        board = list(state)
        # Calcul du résultat en jeu parfait [cite: 21, 23]
        outcome = minimax(board, True)
        
        row = encode_board(board)
        row.append(1 if outcome == 1 else 0) # x_wins [cite: 21]
        row.append(1 if outcome == 0 else 0) # is_draw [cite: 21]
        data.append(row)
    
    # Nommage des colonnes [cite: 19, 21]
    cols = []
    for i in range(9):
        cols.extend([f'c{i}_x', f'c{i}_o'])
    cols.extend(['x_wins', 'is_draw'])
    
    df = pd.DataFrame(data, columns=cols)
    df.to_csv('ressources/dataset.csv', index=False)
    print(f"Dataset généré : {len(df)} lignes.")

 create_csv()

IndentationError: unindent does not match any outer indentation level (<string>, line 27)